# rustwood vs LightGBM — Colab demo

[`rustwood`](https://github.com/advpropsys/rustwood) is a GPU oblivious-tree gradient
booster whose CUDA kernels are pure Rust (compiled to PTX via cuda-oxide). It also has a
GPU-free `--device cpu` trainer and an instant `.rwood` model format.

This notebook trains rustwood and LightGBM on an sklearn dataset and compares **training
time, accuracy, and model save/load speed**.

**Requirements:** a GPU runtime (`Runtime -> Change runtime type -> GPU`). The one-time
build takes ~10-15 min (it compiles the cuda-oxide backend).


## 1. Setup (token, toolchain, build)

`advpropsys/rustwood` is private, so paste a GitHub token with read access below — or make
the repo public and leave it blank.


In [ ]:
GITHUB_TOKEN = ""  #@param {type:"string"}
REPO = "advpropsys/rustwood"


In [ ]:
import os, subprocess, time

# Rust toolchain (the repo pins the exact nightly via rust-toolchain.toml).
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none >/dev/null 2>&1
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']

# Clone (recursive: pulls the cuda-oxide submodule).
url = f'https://{GITHUB_TOKEN}@github.com/{REPO}.git' if GITHUB_TOKEN else f'https://github.com/{REPO}.git'
subprocess.run(['rm','-rf','rustwood'])
assert subprocess.run(['git','clone','--recursive','-q',url,'rustwood']).returncode == 0, 'clone failed (token? repo private?)'
os.chdir('rustwood')

# Detect the GPU's compute capability and build for it (Colab is usually a T4 = sm_75).
cap = subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split('\n')[0].strip()
ARCH = 'sm_' + cap.replace('.', '')
print('GPU compute capability', cap, '->', ARCH)


In [ ]:
# Build the cuda-oxide CLI, then rustwood (this is the slow cell, ~10-15 min).
t = time.time()
!cd external/cuda-oxide && cargo build -q -p cargo-oxide
!ARCH={ARCH} CUDA_PATH=/usr/local/cuda ./build.sh 2>&1 | tail -3
print(f'\nbuild finished in {time.time()-t:.0f}s')
assert os.path.exists('target/release/rustwood'), 'build failed'


## 2. Data (sklearn California Housing)


In [ ]:
import numpy as np, json
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
X = X.astype(np.float32); y = y.astype(np.float32)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
nf = X.shape[1]

# rustwood reads a directory of little-endian f32 blobs + meta.json.
D = '/content/data'; os.makedirs(D, exist_ok=True)
np.ascontiguousarray(Xtr).tofile(f'{D}/x_train.bin'); ytr.tofile(f'{D}/y_train.bin')
np.ascontiguousarray(Xte).tofile(f'{D}/x_test.bin');  yte.tofile(f'{D}/y_test.bin')
json.dump({'n_train': len(ytr), 'n_test': len(yte), 'n_features': nf}, open(f'{D}/meta.json','w'))
print('train', Xtr.shape, 'test', Xte.shape)


## 3. Train rustwood (GPU) + save/load a `.rwood` model


In [ ]:
import re
TREES, DEPTH, LR = 500, 6, 0.1

def rustwood(extra):
    out = subprocess.run(['./target/release/rustwood','--data',D,'--objective','l2',
        '--trees',str(TREES),'--depth',str(DEPTH),'--lr',str(LR)] + extra,
        capture_output=True, text=True)
    return out.stdout

o = rustwood(['--device','gpu','--save-model','/content/m.rwood'])
kv = dict(t.split('=') for t in [l for l in o.splitlines() if l.startswith('RESULT')][0].split()[1:])
rw_train, rw_r2 = float(kv['train_s']), float(kv['r2'])
rw_save = float(re.search(r'bytes, ([0-9.]+) ms', o).group(1))
rw_load = float(re.search(r'load_ms=([0-9.]+)', rustwood(['--load-model','/content/m.rwood'])).group(1))
print(f'rustwood  train={rw_train:.3f}s  R2={rw_r2:.4f}  save={rw_save:.3f}ms  load={rw_load:.3f}ms')


## 4. Train LightGBM


In [ ]:
import lightgbm as lgb, time
from sklearn.metrics import r2_score

m = lgb.LGBMRegressor(n_estimators=TREES, max_depth=DEPTH, num_leaves=2**DEPTH,
                      learning_rate=LR, verbose=-1)
t = time.perf_counter(); m.fit(Xtr, ytr); lgb_train = time.perf_counter() - t
lgb_r2 = r2_score(yte, m.predict(Xte))

def tm(fn, reps=10):
    fn(); s = time.perf_counter_ns()
    for _ in range(reps): fn()
    return (time.perf_counter_ns() - s) / reps / 1e6
lgb_save = tm(lambda: m.booster_.save_model('/content/m.txt'))
lgb_load = tm(lambda: lgb.Booster(model_file='/content/m.txt'))
print(f'LightGBM  train={lgb_train:.3f}s  R2={lgb_r2:.4f}  save={lgb_save:.3f}ms  load={lgb_load:.3f}ms')


## 5. Comparison


In [ ]:
import matplotlib.pyplot as plt
import os
rw_sz = os.path.getsize('/content/m.rwood')/1024
lg_sz = os.path.getsize('/content/m.txt')/1024
print(f'{"":10}{"train_s":>9}{"R2":>8}{"save_ms":>9}{"load_ms":>9}{"size_KB":>9}')
print(f'{"rustwood":10}{rw_train:>9.3f}{rw_r2:>8.4f}{rw_save:>9.3f}{rw_load:>9.3f}{rw_sz:>9.0f}')
print(f'{"LightGBM":10}{lgb_train:>9.3f}{lgb_r2:>8.4f}{lgb_save:>9.3f}{lgb_load:>9.3f}{lg_sz:>9.0f}')

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
labels = ['rustwood', 'LightGBM']; col = ['#E8613C', '#5FA08C']
ax[0].bar(labels, [rw_train, lgb_train], color=col); ax[0].set_title('train time (s)')
ax[1].bar(labels, [rw_load, lgb_load], color=col); ax[1].set_title('model load (ms)'); ax[1].set_yscale('log')
ax[2].bar(labels, [rw_r2, lgb_r2], color=col); ax[2].set_title('test R2'); ax[2].set_ylim(0.7, 0.9)
for a in ax: a.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Notes

- rustwood uses **oblivious (symmetric) trees**, so on this all-numeric dataset LightGBM's
  leaf-wise trees may edge it on accuracy — the structural trade-off. rustwood wins on
  **training speed**, and the `.rwood` format loads **~100x faster** (raw binary vs text).
- The whole library is ~3.2k lines of Rust (vs XGBoost 87k / LightGBM 63k C++/CUDA).
- For a GPU-free run (e.g. a CPU Colab runtime), train with `--device cpu`.
